# 🚀 Qwen2.5-7B-Instruct GGUF Model Deployment

**Model**: Qwen2.5-7B-Instruct-Q4_K_M
**Format**: GGUF
**Port**: 8010

---

In [ ]:
# ===================================================
# 1. INSTALL DEPENDENCIES
# ===================================================

!pip install huggingface-hub -q
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python -q
!pip install fastapi uvicorn nest-asyncio python-multipart -q

# Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("✅ All dependencies installed!")

In [ ]:
# ===================================================
# 2. AUTHENTICATE WITH HUGGING FACE
# ===================================================

from huggingface_hub import login

# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN = "your_huggingface_token_here"  # ⚠️ REPLACE WITH YOUR TOKEN

if HF_TOKEN == "your_huggingface_token_here":
    print("⚠️ Please set your Hugging Face token above")
    login()
else:
    login(token=HF_TOKEN)
    print("✅ Logged in to Hugging Face")

In [ ]:
# ===================================================
# 3. DOWNLOAD THE GGUF MODEL
# ===================================================

from huggingface_hub import hf_hub_download
import os

# ⚠️ REPLACE WITH YOUR REPOSITORY
REPO_ID = "your-username/qwen2.5-7b-instruct-gguf"
MODEL_FILE = "qwen2.5-7b-instruct-q4_k_m.gguf"

os.makedirs("/content/models", exist_ok=True)

print(f"📁 Downloading model from: {REPO_ID}")
print("=" * 60)

try:
    model_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=MODEL_FILE,
        local_dir="/content/models",
        local_dir_use_symlinks=False
    )
    print(f"✅ Downloaded: {MODEL_FILE}")
    print(f"📂 Path: {model_path}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n💡 Try uploading the model to Google Drive instead.")
    model_path = None

In [ ]:
# ===================================================
# 4. CREATE server.py
# ===================================================

%%writefile /content/server.py

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, List
from llama_cpp import Llama
import uvicorn
import json
import os
import time

# ===================================================
# Initialize FastAPI
# ===================================================
app = FastAPI(title="Qwen2.5 GGUF API", description="API for Qwen2.5-7B-Instruct GGUF model")

# ===================================================
# CORS Middleware
# ===================================================
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ===================================================
# Request Models
# ===================================================
class GenerateRequest(BaseModel):
    prompt: str
    system_prompt: Optional[str] = None
    temperature: float = 0.7
    max_tokens: int = 1024
    top_p: float = 0.95
    stop: Optional[List[str]] = None

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[ChatMessage]
    temperature: float = 0.7
    max_tokens: int = 1024
    top_p: float = 0.95

# ===================================================
# Global Variables
# ===================================================
llm = None
MODEL_LOADED = False
MODEL_NAME = "Qwen2.5-7B-Instruct-Q4_K_M"
MODEL_PATH = "/content/models/qwen2.5-7b-instruct-q4_k_m.gguf"

# ===================================================
# Load Model Function
# ===================================================
def load_model():
    global llm, MODEL_LOADED
    
    print("🔄 Loading Qwen2.5-7B-Instruct GGUF model...")
    start_time = time.time()
    
    try:
        # Check if model exists
        if not os.path.exists(MODEL_PATH):
            model_dir = "/content/models"
            if os.path.exists(model_dir):
                gguf_files = [f for f in os.listdir(model_dir) if f.endswith('.gguf')]
                if gguf_files:
                    global MODEL_PATH
                    MODEL_PATH = os.path.join(model_dir, gguf_files[0])
                    print(f"📁 Found model: {MODEL_PATH}")
                else:
                    print("❌ No GGUF model found in /content/models")
                    return False
            else:
                print("❌ Models directory not found: /content/models")
                return False
        
        # Load the model
        llm = Llama(
            model_path=MODEL_PATH,
            n_ctx=8192,
            n_gpu_layers=35,
            n_threads=8,
            n_batch=512,
            flash_attn=True,
            verbose=False
        )
        
        MODEL_LOADED = True
        load_time = time.time() - start_time
        print(f"✅ Model loaded in {load_time:.2f} seconds!")
        print(f"📊 Model: {MODEL_PATH}")
        return True
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        MODEL_LOADED = False
        return False

# ===================================================
# Health Check Endpoint
# ===================================================
@app.get("/")
async def root():
    return {
        "message": "Qwen2.5 GGUF API is running!",
        "model": MODEL_NAME,
        "model_loaded": MODEL_LOADED,
        "status": "healthy" if MODEL_LOADED else "model_not_loaded",
        "endpoints": {
            "/health": "GET - Health check",
            "/generate": "POST - Generate text from prompt",
            "/chat": "POST - Chat with model"
        }
    }

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model_loaded": MODEL_LOADED,
        "model_name": MODEL_NAME,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }

# ===================================================
# Generate Endpoint
# ===================================================
@app.post("/generate")
async def generate(request: GenerateRequest):
    if not MODEL_LOADED or llm is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    try:
        messages = []
        if request.system_prompt:
            messages.append({"role": "system", "content": request.system_prompt})
        messages.append({"role": "user", "content": request.prompt})
        
        output = llm.create_chat_completion(
            messages=messages,
            max_tokens=request.max_tokens,
            temperature=request.temperature,
            top_p=request.top_p,
            stop=request.stop,
            stream=False
        )
        
        return {
            "response": output["choices"][0]["message"]["content"],
            "usage": output.get("usage", {}),
            "model": MODEL_NAME
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/chat")
async def chat(request: ChatRequest):
    if not MODEL_LOADED or llm is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    try:
        messages = [{"role": msg.role, "content": msg.content} for msg in request.messages]
        
        output = llm.create_chat_completion(
            messages=messages,
            max_tokens=request.max_tokens,
            temperature=request.temperature,
            top_p=request.top_p,
            stream=False
        )
        
        return {
            "response": output["choices"][0]["message"]["content"],
            "usage": output.get("usage", {}),
            "model": MODEL_NAME
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8010)

In [ ]:
# ===================================================
# 5. LOAD AND TEST THE MODEL
# ===================================================

import server

# Load the model
server.load_model()

# Test the model
if server.MODEL_LOADED:
    print("\n🧪 Testing model...")
    try:
        output = server.llm.create_chat_completion(
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "What is the capital of France?"}
            ],
            max_tokens=50,
            temperature=0.7
        )
        print(f"🤖 Response: {output['choices'][0]['message']['content']}")
        print("\n✅ Model is working!")
    except Exception as e:
        print(f"❌ Test failed: {e}")
else:
    print("❌ Model not loaded")

In [ ]:
# ===================================================
# 6. START SERVER ON PORT 8010
# ===================================================

import subprocess
import time
import signal

print("🚀 Starting API Server on port 8010...")
print("=" * 60)

# Kill any existing process on port 8010
!fuser -k 8010/tcp 2>/dev/null || true
time.sleep(2)

# Start server with nohup
!nohup uvicorn server:app --host 0.0.0.0 --port 8010 > server.log 2>&1 &

time.sleep(5)

# Check if server is running
import requests
try:
    response = requests.get("http://localhost:8010/health", timeout=5)
    print(f"✅ Server is running on port 8010")
    print(f"📊 Health: {response.json()}")
except Exception as e:
    print(f"⚠️ Server check: {e}")
    print("📋 Checking server.log...")
    !cat server.log

print("\n" + "=" * 60)
print("📍 Server running at: http://localhost:8010")
print("=" * 60)

In [ ]:
# ===================================================
# 7. START CLOUDFLARE TUNNEL
# ===================================================

print("🌐 Starting Cloudflare Tunnel...")
print("=" * 60)
print("🔗 YOUR PUBLIC URL WILL APPEAR BELOW")
print("=" * 60)
print("")

# Run cloudflared tunnel
!./cloudflared-linux-amd64 tunnel --url http://localhost:8010